# Return vs Risk Frontier — Binned Density per Block

Plots the classic mean–variance plane with millions of real DeFi wallets.

**Scaling (monthly, 30-day):**
- `vol_monthly = sqrt(variance_daily) * sqrt(30)` → expressed as %
- `ret_monthly = expected_return_daily * 30` → expressed as %

In [9]:
import duckdb
import numpy as np
import time

In [10]:
# --- CONFIG ---
import sys
sys.path.insert(0, "../../shared")
from locations import Location

ROOT = str(Location.MPT_DATA)
OUT_CSV = "return_vs_risk_binned.csv"

# Pick your blocks here
SELECTED_BLOCKS = [
    9193266,   # Jan 2020
    13330090,  # Oct 2021
    15053226,  # Jul 2022
    18037988,  # Sep 2023
    20207949,  # Jul 2024
    23914921,  # Dec 2025
]

# Binning parameters (monthly % scale)
# Volatility: 0% to 200%, step 1pp
VOL_MIN, VOL_MAX, VOL_STEP = 0.0, 200.0, 2.0
# Return: -50% to +50%, step 1pp
RET_MIN, RET_MAX, RET_STEP = -100.0, 100.0, 2.0

print(f"Blocks selected: {len(SELECTED_BLOCKS)}")
print(f"Vol bins: {int((VOL_MAX - VOL_MIN) / VOL_STEP)}, Return bins: {int((RET_MAX - RET_MIN) / RET_STEP)}")

Blocks selected: 6
Vol bins: 100, Return bins: 100


In [11]:
con = duckdb.connect(database=':memory:')

con.execute(f"""
    CREATE VIEW all_data AS
    SELECT * FROM read_parquet('{ROOT}/*/*.parquet', hive_partitioning=true)
""")

# Verify blocks
block_list = ', '.join(str(b) for b in SELECTED_BLOCKS)
info = con.execute(f"""
    SELECT block_number, date, COUNT(*) AS n_wallets
    FROM all_data
    WHERE block_number IN ({block_list})
    GROUP BY block_number, date
    ORDER BY block_number
""").fetchdf()

display(info)
print(f"Total wallets to process: {info['n_wallets'].sum():,}")

,block_number,date,n_wallets
0,9193266,2020-01-01,879029
1,13330090,2021-10-01,2386807
2,15053226,2022-07-01,3104578
3,18037988,2023-09-01,3678201
4,20207949,2024-07-01,4583434
5,23914921,2025-12-01,7112115


Total wallets to process: 21,744,164


In [12]:
# --- COMPUTE & BIN ---
t0 = time.time()
print("Computing monthly return and volatility, binning into 2D grid ...")

df_binned = con.execute(f"""
    WITH metrics AS (
        SELECT
            block_number,
            date,
            -- Monthly scaling
            SQRT(initial_variance_daily) * SQRT(30) * 100  AS vol_monthly_pct,
            initial_expected_return_daily * 30 * 100        AS ret_monthly_pct
        FROM all_data
        WHERE block_number IN ({block_list})
          AND initial_variance_daily > 0
          AND initial_variance_daily IS NOT NULL
          AND initial_expected_return_daily IS NOT NULL
    ),
    binned AS (
        SELECT
            block_number,
            date,
            -- Clamp and bin volatility
            FLOOR(LEAST(GREATEST(vol_monthly_pct, {VOL_MIN}), {VOL_MAX} - 0.001) / {VOL_STEP}) * {VOL_STEP} AS vol_bin,
            -- Clamp and bin return
            FLOOR(LEAST(GREATEST(ret_monthly_pct, {RET_MIN}), {RET_MAX} - 0.001) / {RET_STEP}) * {RET_STEP} AS ret_bin
        FROM metrics
    )
    SELECT
        block_number,
        date,
        ROUND(vol_bin, 2)  AS vol_bin,
        ROUND(ret_bin, 2)  AS ret_bin,
        COUNT(*)           AS n_wallets
    FROM binned
    GROUP BY block_number, date, vol_bin, ret_bin
    ORDER BY block_number, vol_bin, ret_bin
""").fetchdf()

print(f"Done in {time.time() - t0:.1f}s")
print(f"Result: {len(df_binned):,} bins across {df_binned['block_number'].nunique()} blocks")
print(f"Total wallets represented: {df_binned['n_wallets'].sum():,}")

Computing monthly return and volatility, binning into 2D grid ...
Done in 0.9s
Result: 38,826 bins across 6 blocks
Total wallets represented: 21,744,164


In [13]:
# --- PREVIEW ---
print("Bin counts per block:")
summary = df_binned.groupby(['block_number', 'date']).agg(
    n_bins=('n_wallets', 'count'),
    total_wallets=('n_wallets', 'sum')
).reset_index()
display(summary)

print("\nTop-populated bins:")
display(df_binned.nlargest(15, 'n_wallets'))

Bin counts per block:


,block_number,date,n_bins,total_wallets
0,9193266,2020-01-01,3855,879029
1,13330090,2021-10-01,5795,2386807
2,15053226,2022-07-01,5776,3104578
3,18037988,2023-09-01,7647,3678201
4,20207949,2024-07-01,7710,4583434
5,23914921,2025-12-01,8043,7112115



Top-populated bins:


,block_number,date,vol_bin,ret_bin,n_wallets
30786,23914921,2025-12-01,0.0,0.0,805691
23775,20207949,2024-07-01,32.0,8.0,198262
4351,13330090,2021-10-01,30.0,-6.0,193588
10489,15053226,2022-07-01,40.0,-40.0,187717
16303,18037988,2023-09-01,36.0,20.0,180579
31383,23914921,2025-12-01,28.0,-22.0,175106
1332,9193266,2020-01-01,60.0,-22.0,171085
23074,20207949,2024-07-01,0.0,-2.0,144334
15428,18037988,2023-09-01,0.0,-2.0,136694
31537,23914921,2025-12-01,32.0,-22.0,123413


In [14]:
# --- QUICK STATS per block ---
for block in SELECTED_BLOCKS:
    sub = df_binned[df_binned['block_number'] == block]
    if sub.empty:
        print(f"\nBlock {block}: no data")
        continue
    total = sub['n_wallets'].sum()
    date = sub['date'].iloc[0]

    avg_vol = np.average(sub['vol_bin'] + VOL_STEP / 2, weights=sub['n_wallets'])
    avg_ret = np.average(sub['ret_bin'] + RET_STEP / 2, weights=sub['n_wallets'])

    # % with positive expected return
    pct_pos = sub[sub['ret_bin'] >= 0]['n_wallets'].sum() / total * 100

    # % in low-vol region (<=50%)
    pct_low_vol = sub[sub['vol_bin'] <= 50]['n_wallets'].sum() / total * 100

    print(f"\nBlock {block} ({date}): {total:,} wallets")
    print(f"  Avg monthly return: {avg_ret:+.2f}%")
    print(f"  Avg monthly vol:    {avg_vol:.1f}%")
    print(f"  Positive return:    {pct_pos:.1f}%")
    print(f"  Low vol (<=50%):    {pct_low_vol:.1f}%")


Block 9193266 (2020-01-01 00:00:00): 879,029 wallets
  Avg monthly return: -19.65%
  Avg monthly vol:    43.5%
  Positive return:    5.1%
  Low vol (<=50%):    70.2%

Block 13330090 (2021-10-01 00:00:00): 2,386,807 wallets
  Avg monthly return: +4.01%
  Avg monthly vol:    35.2%
  Positive return:    57.3%
  Low vol (<=50%):    91.8%

Block 15053226 (2022-07-01 00:00:00): 3,104,578 wallets
  Avg monthly return: -30.97%
  Avg monthly vol:    38.9%
  Positive return:    3.6%
  Low vol (<=50%):    86.7%

Block 18037988 (2023-09-01 00:00:00): 3,678,201 wallets
  Avg monthly return: -3.39%
  Avg monthly vol:    25.6%
  Positive return:    26.8%
  Low vol (<=50%):    92.3%

Block 20207949 (2024-07-01 00:00:00): 4,583,434 wallets
  Avg monthly return: -5.92%
  Avg monthly vol:    26.9%
  Positive return:    29.4%
  Low vol (<=50%):    93.5%

Block 23914921 (2025-12-01 00:00:00): 7,112,115 wallets
  Avg monthly return: -17.93%
  Avg monthly vol:    29.7%
  Positive return:    14.6%
  Low vol 

In [15]:
# --- SAVE ---
df_binned.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}  ({len(df_binned):,} rows)")

Saved: return_vs_risk_binned.csv  (38,826 rows)


In [16]:
con.close()